<a href="https://colab.research.google.com/github/DimDragg/-/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0_%D1%80%D0%BE%D0%B1%D0%BE%D1%82%D0%B0_6_%D0%9A%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F_%D0%97%D0%B0%D0%BF%D0%BE%D0%B1%D1%96%D0%B3%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BF%D0%B5%D1%80%D0%B5%D0%BD%D0%B0%D0%B2%D1%87%D0%B0%D0%BD%D0%BD%D1%8E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Лабораторна робота 6.Класифікація. Запобігання перенавчанню. Чуркін ФІТ 3-15 номер 15

Був присутній на парі

Завдання

Завантажити датасет https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset?select=diabetes_prediction_dataset.csv Провести попередній аналіз і підготовку даних. Перевірити наявність пропущених даних, дублікатив, попарну кореляцію, кореляцію з цільовою змінною. Перевірити симетричність розподілу, балансування класів. Побудувати моделі models = { 'LogisticRegression': LogisticRegression(), 'RidgeClassifier': RidgeClassifier(), 'SGDClassifier': SGDClassifier(), 'SVC': SVC() }

Вивести класифікаційний звіт, матрицю плутанини. Підібрати оптимальні параметри HalvingGridSearchCV Вивести оптимальні параметри Вивести класифікаційний звіт Вивести 10 випадкових записів з тестової вибірки, справжній і прогнозований клас. Написати висновки.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("diabetes_prediction_dataset.csv.zip", compression='zip')

print(df.head())
print(df.info())

print("\nПерепустки:")
print(df.isnull().sum())

print("\nДублікати:", df.duplicated().sum())
df = df.drop_duplicates()

# Перетворення категоріальних змінних
categorical_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Кореляція
corr = df.corr()
print("\nКореляція с diabetes:")
print(corr["diabetes"].sort_values(ascending=False))

# Баланс класів
print("\nБаланс класів:")
print(df["diabetes"].value_counts())

# Поділ на train/test
from sklearn.model_selection import train_test_split

X = df.drop("diabetes", axis=1)
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Масштабування
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Базові моделі
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RidgeClassifier': RidgeClassifier(),
    'SGDClassifier': SGDClassifier(),
    'SVC_linear': SVC(kernel='linear')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print(f"\n{name}")
    print(classification_report(y_test, preds))
    print("Матрица ошибок:")
    print(confusion_matrix(y_test, preds))

# Пошук по сітці з HalvingGridSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear']
}

grid = HalvingGridSearchCV(SVC(), param_grid, cv=3)
grid.fit(X_train, y_train)

print("\nНайкращі параметри:")
print(grid.best_params_)

best_model = grid.best_estimator_
best_preds = best_model.predict(X_test)

print("\nЗвіт найкращої моделі:")
print(classification_report(y_test, best_preds))
print("Матрица помилок:")
print(confusion_matrix(y_test, best_preds))

# 10 випадкових прикладів
sample_idx = np.random.choice(len(X_test), 10, replace=False)
print("\n10 випадкових прикладів:")

for i in sample_idx:
    print(f"Реальний: {y_test.values[i]} | Прогноз: {best_preds[i]}")

   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  
3          5.0                  155         0  
4          4.8                  155         0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age               

Висновок

У ході виконання лабораторної роботи було проведено аналіз датасету для задачі класифікації на прикладі прогнозування наявності діабету. Було виконано попередню обробку даних, включаючи перевірку пропущених значень, видалення дублікатів та кодування категоріальних ознак.

Побудовано декілька моделей класифікації: Logistic Regression, Ridge Classifier, SGD Classifier та SVC. Для кожної моделі було обчислено метрики якості, зокрема precision, recall та f1-score, а також побудовано матриці плутанини.

З метою покращення якості моделі було застосовано метод підбору гіперпараметрів HalvingGridSearchCV, що дозволило знайти оптимальні параметри та зменшити перенавчання. Найкращі результати показала модель (вставишь свою), яка забезпечила найвищу точність класифікації.